# Notebook 06 — Training-only attention audit definitions

This companion provides a bounded audit of the current attention approximation before the three-hour experiment chooses its fixed layerwise feature allocation. Inputs are the completed Notebook 05 learning checkpoint and four deterministic training images; no development or held-out images enter selection. Outputs are sampled output/Jacobian errors, normalized weight-error bounds, and a fixed-total feature recommendation.

Expected audit runtime: under 10 minutes on GPU 0, enforced by its original deadline. This definition notebook itself performs no training or GPU work when run top to bottom. The Notebook 06 worker calls `audit` within the shared three-hour clock. Checkpoints: read-only `checkpoints/learning-resolution-stable-001/stages/learning/arms/s20260928-semantic/`. Diagnostic captures and partial measurements live in the new run's `attention_audit/` directory. Resume by calling `audit` with the identical root, artifact directory, inputs and original outer deadline; completed outputs are checksum-verified and reused, partial work keeps its original audit deadline.

In [ ]:
from pathlib import Path
import os, json
ROOT = Path(os.environ.get("FMM_JEPA_ROOT", "/home/benjamin/workspace/ahr_go/fmm_jeppa")).resolve()
_boot = {}
for _cell in json.loads((ROOT / "fmm_jepa/bootstrap.ipynb").read_text())["cells"]:
    if _cell["cell_type"] == "code" and "ahr-export" in _cell.get("metadata", {}).get("tags", []):
        exec(compile("".join(_cell["source"]), "bootstrap.ipynb#ahr-export", "exec"), _boot)
_boot["install_notebook_package"](ROOT)


## Method and selection rule

Compare exact dense softmax with the unchanged local-exact/far-random-feature operator, using ranks 16, 32, 48 and 64, with and without differentiable mean-query-plus-mean-key centering. The receiver set is bounded, but all valid sources and all input derivatives remain present. `epsilon = sum(abs(approx_weight - exact_weight)) / sum(exact_weight)` yields an output bound `2 M epsilon / (1 - epsilon)` whenever `epsilon < 1`. Ineligible rows have no asserted finite bound.

The score gives equal weight to relative output error and one random-cotangent Jacobian error. Starting from 16 features per layer, four greedy increments of 16 allocate exactly 128 total features. An allocation replaces uniform 32 only if its score improves by at least 2%; otherwise uniform capacity is retained. Selection uses training images only and is an exploratory operator proxy, never a classification conclusion.

In [ ]:
"""Training-only, checkpoint-grounded audit of sampled attention operators."""
from pathlib import Path
import copy, gc, hashlib, json, math, sys, time, types
import numpy as np
import torch
from fmm_jepa.training import atomic_json, digest_file, run_writer
from fmm_jepa.food101_training import validate_training_manifest, isolated_rng
from fmm_jepa.food101_data import Food101Adapter, _read_sealed, make_batch
from fmm_jepa.semantic_attention import _orthogonal_gaussian_features
from fmm_jepa.semantic_evaluation import _build_semantic_comparator

SOURCE_RUN = "learning-resolution-stable-001"
SOURCE_STAGE = "learning"
SOURCE_SEED = 20260928
AUDIT_SCHEMA = "fmm-jepa-attention-math-audit/v1"
RANKS = (16, 32, 48, 64)
ALLOCATION_RULE = {
    "ranks": list(RANKS), "total_features_across_four_layers": 128,
    "score": "0.5 * (relative_output_L2 + relative_random_cotangent_Jacobian_L2), mean over heads",
    "algorithm": "Start 16 per layer; four increments of 16 maximize immediate score reduction; ties select lower layer index",
    "acceptance": "Adopt only if centered allocation summed layer score is at least 2 percent below centered uniform32; otherwise retain uniform32",
    "selection_data": "Four fixed training images, no development labels or images; exploratory operator proxy, not validation accuracy",
}


def _digest(value):
    return hashlib.sha256(json.dumps(value, sort_keys=True, separators=(",", ":")).encode()).hexdigest()


def _export_digest(path):
    document = json.loads(Path(path).read_text())
    return _digest(["".join(c["source"]) for c in document["cells"]
                    if c["cell_type"] == "code" and "ahr-export" in c.get("metadata", {}).get("tags", [])])


def _load_stable(root):
    name = "fmm05_stable_attention"
    if name in sys.modules:
        return sys.modules[name]
    path = Path(root) / "experiments/05_scaling/stable_attention.ipynb"
    module = types.ModuleType(name)
    module.__file__ = str(path)
    sys.modules[name] = module
    try:
        for cell in json.loads(path.read_text())["cells"]:
            if cell["cell_type"] == "code" and "ahr-export" in cell.get("metadata", {}).get("tags", []):
                exec(compile("".join(cell["source"]), str(path) + "#ahr-export", "exec"), module.__dict__)
    except BaseException:
        sys.modules.pop(name, None)
        raise
    return module


def _check_deadline(deadline, reserve=0):
    if time.time() + reserve >= deadline:
        raise TimeoutError("Mathematical attention audit exceeded its bounded deadline")


def _relative_norm(error, reference):
    # Diagnostic normalizer only; never applied to attention weights or outputs.
    return float((error.norm() / reference.norm().clamp_min(1e-20)).detach())


def _sampled_weights(q, k, v, indices, valid, grid, omega, leaf_size, local_radius,
                     receivers, centered=False, exact_only=False):
    """Independent FP64 dense source oracle, bounded to sampled receivers.

    The center is differentiated through both query/key means. Selected output
    Jacobians therefore include the dependence on every valid query/key input.
    No approximation or stabilization clamps enter the represented operator.
    """
    mask = valid[:, None, :, None]
    q, k, v = [torch.where(mask, t.double(), torch.zeros_like(t, dtype=torch.float64)) for t in (q, k, v)]
    if centered:
        denominator = valid.sum(-1).clamp_min(1)[:, None, None, None]
        center = (q.sum(-2, keepdim=True) + k.sum(-2, keepdim=True)) / denominator
        k = torch.where(mask, k - center, torch.zeros_like(k))
    gather = receivers[:, None, :, None].expand(-1, q.shape[1], -1, q.shape[-1])
    qr = q.gather(-2, gather)
    exact_log = torch.matmul(qr, k.transpose(-2, -1)) / math.sqrt(q.shape[-1])
    exact_log = exact_log.masked_fill(~valid[:, None, None, :], -torch.inf)
    exact_weights = torch.softmax(exact_log, dim=-1)
    if exact_only:
        return torch.matmul(exact_weights, v), {"weights": exact_weights, "exact_log": exact_log}
    scale, rank = q.shape[-1] ** -0.25, omega.shape[-2]
    qs, ks = qr * scale, k * scale
    qlog = torch.einsum("bhrd,hmd->bhrm", qs, omega.double()) - qs.square().sum(-1, keepdim=True) / 2 - math.log(rank) / 2
    klog = torch.einsum("bhnd,hmd->bhnm", ks, omega.double()) - ks.square().sum(-1, keepdim=True) / 2 - math.log(rank) / 2
    approx_log = torch.logsumexp(qlog.unsqueeze(-2) + klog.unsqueeze(-3), dim=-1)
    cell_rows = indices.clamp_min(0).div(int(grid[1]), rounding_mode="floor").div(leaf_size, rounding_mode="floor")
    cell_cols = indices.clamp_min(0).remainder(int(grid[1])).div(leaf_size, rounding_mode="floor")
    receiver_rows = cell_rows.gather(1, receivers)
    receiver_cols = cell_cols.gather(1, receivers)
    local = ((receiver_rows[:, :, None] - cell_rows[:, None, :]).abs() <= local_radius) & ((receiver_cols[:, :, None] - cell_cols[:, None, :]).abs() <= local_radius)
    hybrid_log = torch.where(local[:, None], exact_log, approx_log).masked_fill(~valid[:, None, None], -torch.inf)
    weights = torch.softmax(hybrid_log, -1)
    output = torch.matmul(weights, v)
    shift = exact_log.amax(-1, keepdim=True)
    exact_raw = torch.exp(exact_log - shift)
    hybrid_raw = torch.exp(hybrid_log - shift)
    epsilon = (hybrid_raw - exact_raw).abs().sum(-1) / exact_raw.sum(-1)
    far_mass = (exact_weights * (~local[:, None])).sum(-1)
    max_value_norm = v.norm(dim=-1).amax(-1, keepdim=True)
    eligible = epsilon < 1
    # Ineligible bounds are represented as null in JSON, never as a finite guarantee.
    bound = torch.where(eligible, 2 * max_value_norm * epsilon / (1 - epsilon), torch.full_like(epsilon, float("inf")))
    return output, {"weights": weights, "epsilon": epsilon, "bound": bound,
                    "bound_eligible": eligible, "far_mass": far_mass}


def _receiver_indices(valid, count):
    result = []
    for row in valid:
        positions = row.nonzero(as_tuple=False).flatten()
        if len(positions) < count:
            raise ValueError("Insufficient valid receiver positions for fixed audit count")
        result.append(positions[torch.linspace(0, len(positions)-1, count, device=row.device).round().long()])
    return torch.stack(result)


def _compare_capture(capture, rank, centered, receiver_count, device):
    q, k, v = [capture[name].to(device).double().requires_grad_(True) for name in ("q", "k", "v")]
    indices, valid = capture["indices"].to(device), capture["valid"].to(device)
    receivers = _receiver_indices(valid, receiver_count)
    omega = _orthogonal_gaussian_features(q.shape[1], rank, q.shape[-1], capture["feature_seed"]).to(device)
    common = (q, k, v, indices, valid, capture["grid"], omega, capture["leaf_size"], capture["local_radius"], receivers)
    exact, _ = _sampled_weights(*common, exact_only=True)
    approximate, details = _sampled_weights(*common, centered=centered)
    # One fixed Rademacher cotangent per layer, identical across every candidate.
    generator = torch.Generator(device="cpu").manual_seed(20260930 + capture["layer"])
    cotangent = (2 * torch.randint(0, 2, exact.shape, generator=generator) - 1).to(device=device, dtype=torch.float64)
    exact_gradients = torch.autograd.grad((exact * cotangent).sum(), (q, k, v), retain_graph=True)
    approximate_gradients = torch.autograd.grad((approximate * cotangent).sum(), (q, k, v))
    if not all(bool(torch.isfinite(t).all()) for t in (exact, approximate, *exact_gradients, *approximate_gradients)):
        raise FloatingPointError("Nonfinite output/Jacobian in sampled FP64 mathematical audit")
    output_errors = (approximate - exact).detach().norm(dim=-1)
    valid_bound = details["bound_eligible"].detach()
    if bool((output_errors[valid_bound] > details["bound"].detach()[valid_bound] + 1e-9).any()):
        raise AssertionError("Attention-output error exceeded its deterministic normalized-weight bound")
    rows = []
    for head in range(q.shape[1]):
        output_relative = _relative_norm(approximate[:, head] - exact[:, head], exact[:, head])
        eg = torch.cat([g[:, head].flatten() for g in exact_gradients])
        ag = torch.cat([g[:, head].flatten() for g in approximate_gradients])
        gradient_relative = _relative_norm(ag - eg, eg)
        eps = details["epsilon"][:, head].detach().flatten()
        eligible = details["bound_eligible"][:, head].detach().flatten()
        bounds = details["bound"][:, head].detach().flatten()[eligible]
        far = details["far_mass"][:, head].detach().flatten()
        rows.append({"layer": capture["layer"], "head": head, "rank": rank, "centered": centered,
            "relative_output_l2": output_relative, "relative_jacobian_l2": gradient_relative,
            "selection_score": .5 * (output_relative + gradient_relative),
            "epsilon_mean": float(eps.mean()), "epsilon_p95": float(torch.quantile(eps, .95)),
            "epsilon_max": float(eps.max()), "epsilon_below_one_fraction": float(eligible.double().mean()),
            "far_exact_attention_mass_mean": float(far.mean()),
            "far_exact_attention_mass_p95": float(torch.quantile(far, .95)),
            "eligible_bound_max": float(bounds.max()) if len(bounds) else None,
            "bound_violations": 0, "sampled_receivers": len(eps),
            "output_error_l2_max": float(output_errors[:, head].max())})
    return rows


def recommend_feature_counts(rows):
    """Predeclared fixed-total greedy capacity allocation, using training only."""
    scores = {}
    for layer in range(4):
        for rank in RANKS:
            values = [r["selection_score"] for r in rows if r["layer"] == layer and r["rank"] == rank and r["centered"]]
            if not values or not all(math.isfinite(x) for x in values):
                raise ValueError("Complete finite centered layer/rank scores are required")
            scores[(layer, rank)] = float(np.mean(values))
    counts, path = [16] * 4, []
    while sum(counts) < 128:
        candidates = [(scores[(layer, counts[layer])] - scores[(layer, counts[layer] + 16)], layer)
                      for layer in range(4) if counts[layer] < 64]
        gain, chosen = max(candidates, key=lambda pair: (pair[0], -pair[1]))
        counts[chosen] += 16
        path.append({"layer": chosen, "new_rank": counts[chosen], "marginal_score_reduction": gain})
    uniform = sum(scores[(layer, 32)] for layer in range(4))
    allocated = sum(scores[(layer, counts[layer])] for layer in range(4))
    accepted = allocated <= .98 * uniform and allocated < uniform
    return {"feature_counts": counts if accepted else [32] * 4,
            "candidate_feature_counts": counts, "accepted": accepted,
            "uniform32_score_sum": uniform, "allocated_score_sum": allocated,
            "fractional_score_reduction": (uniform - allocated) / max(uniform, 1e-20),
            "greedy_steps": path, "rule": ALLOCATION_RULE}


def _source_bundle(root):
    prior = Path(root) / "artifacts/notebook_05" / SOURCE_RUN / "stages" / SOURCE_STAGE
    split = json.loads((prior / "split_manifest.json").read_text())
    source = _read_sealed(Path(root) / "data/food101/dataset_manifest.json")
    return {"dataset": Food101Adapter(Path(root) / "data/food101", source),
            "train_ids": split["selected_ids"]["training"],
            "dev_ids": split["selected_ids"]["development"],
            "heldout_ids": split["selected_ids"]["heldout"]}


def audit(root, artifact_dir, bundle=None, deadline_unix=None, device="cuda:0",
          max_seconds=600, sample_count=4, receiver_count=16):
    """Run/reuse a bounded training-only audit; no optimizer updates or probes.

    Writes immutable protocol, captured activations, per-head measurements and
    recommendation under artifact_dir/attention_audit. Partial captures/rows are
    resumable under the same protocol and original deadline. The outer worker
    enforces the absolute three-hour deadline; this call also reserves its own
    maximum of 600 seconds on its initial invocation.
    """
    root, artifact = Path(root).resolve(), Path(artifact_dir).resolve()
    artifact.relative_to(root)
    folder = artifact / "attention_audit"
    folder.mkdir(parents=True, exist_ok=True)
    with run_writer(folder):
        return _audit(root, folder, bundle, deadline_unix, device, int(max_seconds), int(sample_count), int(receiver_count))


def _audit(root, folder, bundle, deadline_unix, device, max_seconds, sample_count, receiver_count):
    started = time.time()
    prior = root / "artifacts/notebook_05" / SOURCE_RUN / "stages" / SOURCE_STAGE
    plan = json.loads((prior / "plan.json").read_text())
    arm = next(a for a in plan["arms"] if a["kind"] == "semantic" and a["seed"] == SOURCE_SEED)
    config = copy.deepcopy(arm["config"])
    checkpoint_folder = root / "checkpoints" / SOURCE_RUN / "stages" / SOURCE_STAGE / "arms" / arm["id"]
    manifest = validate_training_manifest(checkpoint_folder)
    if manifest["successful_updates"] != 1000 or manifest["attempted_updates"] != 1000:
        raise ValueError("Audit requires the clean completed 1000-update source model")
    checkpoint_path = checkpoint_folder / manifest["latest"]
    bundle = _source_bundle(root) if bundle is None else bundle
    train_ids = sorted(bundle["train_ids"])
    generator = np.random.default_rng(20260930)
    chosen = sorted(generator.choice(train_ids, size=sample_count, replace=False).tolist())
    if set(chosen) & (set(bundle.get("dev_ids", [])) | set(bundle.get("heldout_ids", []))):
        raise ValueError("Diagnostic candidate selection crossed a reserved split")
    image_hashes = {sid: digest_file(bundle["dataset"].path_for(sid)) for sid in chosen}
    identity = {"schema": AUDIT_SCHEMA, "source_run": SOURCE_RUN, "source_seed": SOURCE_SEED,
        "source_checkpoint": str(checkpoint_path), "source_sha256": digest_file(checkpoint_path),
        "source_manifest_sha256": digest_file(checkpoint_folder / "manifest.json"),
        "source_plan_sha256": digest_file(prior / "plan.json"), "sample_ids": chosen,
        "image_hashes": image_hashes, "sample_count": sample_count, "receiver_count": receiver_count,
        "device": str(device), "ranks": list(RANKS), "allocation_rule": ALLOCATION_RULE,
        "audit_export_hash": _export_digest(root / "experiments/06_math/diagnostics.ipynb"),
        "stable_export_hash": _export_digest(root / "experiments/05_scaling/stable_attention.ipynb"),
        "torch_version": torch.__version__, "heldout_status": "NOT_RUN", "development_pixels_read": 0,
        "gradient_note": "One fixed Rademacher cotangent; reports a sampled Jacobian projection, not the full Jacobian or training gradients"}
    identity_hash = _digest(identity)
    protocol_path, result_path = folder / "protocol.json", folder / "audit_result.json"
    if protocol_path.exists():
        protocol = json.loads(protocol_path.read_text())
        if protocol["identity_hash"] != identity_hash:
            raise ValueError("Attention audit source or inputs changed; reuse refused")
    else:
        deadline = min(float(deadline_unix) if deadline_unix is not None else math.inf, started + max_seconds)
        protocol = {"identity_hash": identity_hash, "identity": identity, "started_unix": started,
                    "deadline_unix": deadline, "registered_before_capture": True}
        atomic_json(protocol_path, protocol)
    if result_path.exists():
        seal = json.loads((folder / "result_manifest.json").read_text())
        if seal["identity_hash"] != identity_hash or digest_file(result_path) != seal["sha256"]:
            raise ValueError("Completed attention audit checksum changed")
        return json.loads(result_path.read_text())
    deadline = protocol["deadline_unix"]
    if deadline_unix is not None:
        deadline = min(deadline, float(deadline_unix))
    _check_deadline(deadline, 5)
    capture_path, capture_meta = folder / "activations.pt", folder / "activation_manifest.json"
    if capture_path.exists() or capture_meta.exists():
        meta = json.loads(capture_meta.read_text())
        if meta["identity_hash"] != identity_hash or digest_file(capture_path) != meta["sha256"]:
            raise ValueError("Saved attention activations changed")
        captures = torch.load(capture_path, map_location="cpu", weights_only=True)
    else:
        stable = _load_stable(root)
        with isolated_rng(SOURCE_SEED):
            model = _build_semantic_comparator(config, "semantic")
        payload = torch.load(checkpoint_path, map_location="cpu", weights_only=False)
        model.load_state_dict(payload["model"])
        del payload
        model = model.to(device).eval()
        captures = []
        for layer, block in enumerate(model.context.blocks):
            def capture_call(mixer, q, k, v, indices, valid, grid, layer_index=layer):
                captures.append({"layer": layer_index, "q": q.detach().cpu(), "k": k.detach().cpu(),
                    "v": v.detach().cpu(), "indices": indices.detach().cpu(), "valid": valid.detach().cpu(),
                    "grid": tuple(map(int, grid)), "feature_seed": mixer.feature_seed,
                    "leaf_size": mixer.local_leaf_size, "local_radius": mixer.local_radius})
                return stable.stable_mix_heads(mixer, q, k, v, indices, valid, grid)
            block.mixer.mix_heads = types.MethodType(capture_call, block.mixer)
        batch = make_batch(bundle["dataset"], chosen, [0] * len(chosen), config, training=True)
        with torch.no_grad():
            model.context(batch["images"].to(device), batch["masks"]["context_indices"].to(device),
                          batch["masks"]["context_valid"].to(device))
        if len(captures) != 4 or [c["layer"] for c in captures] != list(range(4)):
            raise ValueError("Expected one captured online attention input for each of four layers")
        temporary = capture_path.with_suffix(".pt.tmp")
        torch.save(captures, temporary)
        temporary.replace(capture_path)
        atomic_json(capture_meta, {"identity_hash": identity_hash, "sha256": digest_file(capture_path),
                                 "augmentation_ids": batch["augmentation_ids"], "mask_ids": batch["mask_ids"]})
        del model, batch
        gc.collect()
        if str(device).startswith("cuda"):
            torch.cuda.empty_cache()
    measurements = folder / "measurements"
    measurements.mkdir(exist_ok=True)
    rows = []
    for capture in captures:
        for centered in (False, True):
            for rank in RANKS:
                _check_deadline(deadline, 2)
                path = measurements / ("layer%d_center%d_rank%d.json" % (capture["layer"], centered, rank))
                if path.exists():
                    record = json.loads(path.read_text())
                    if record["identity_hash"] != identity_hash:
                        raise ValueError("Partial audit identity changed")
                    expected = _digest(record["rows"])
                    if expected != record["rows_sha256"]:
                        raise ValueError("Partial audit data checksum changed")
                    part = record["rows"]
                else:
                    part = _compare_capture(capture, rank, centered, receiver_count, device)
                    atomic_json(path, {"identity_hash": identity_hash, "rows": part, "rows_sha256": _digest(part)})
                rows.extend(part)
        print("Attention audit layer %d/4 complete" % (capture["layer"] + 1), flush=True)
    recommendation = recommend_feature_counts(rows)
    summary = []
    for centered in (False, True):
        for rank in RANKS:
            selected = [r for r in rows if r["centered"] == centered and r["rank"] == rank]
            summary.append({"centered": centered, "rank": rank,
                "mean_relative_output_l2": float(np.mean([r["relative_output_l2"] for r in selected])),
                "mean_relative_jacobian_l2": float(np.mean([r["relative_jacobian_l2"] for r in selected])),
                "mean_selection_score": float(np.mean([r["selection_score"] for r in selected]))})
    result = {"schema": AUDIT_SCHEMA, "status": "PASS", "identity_hash": identity_hash,
        "identity": identity, "measurements": rows, "summary": summary, "recommendation": recommendation,
        "feature_counts": recommendation["feature_counts"], "training_optimizer_updates": 0,
        "elapsed_seconds_this_invocation": time.time() - started, "completed_unix": time.time(),
        "bound_violations": sum(r["bound_violations"] for r in rows), "heldout_status": "NOT_RUN",
        "limitations": ["Four training images and one trained seed; exploratory in-sample operator screen",
                        "Captures baseline online encoder context attention; not teacher/full-image attention",
                        "One random cotangent per layer; not a full Jacobian norm estimate",
                        "Dense oracle characterizes mathematical approximation; separate backend validation certifies implementation equivalence",
                        "Better operator error does not guarantee better JEPA loss or classification accuracy"]}
    atomic_json(result_path, result)
    atomic_json(folder / "result_manifest.json", {"identity_hash": identity_hash, "sha256": digest_file(result_path)})
    return result


In [ ]:
print("Audit definitions ready; invoke audit from the bounded Notebook 06 worker.")